# Pinecone Basics

This notebook covers the fundamentals of using Pinecone as a managed vector database.

## Topics
1. Setting up Pinecone with API key
2. Creating an index
3. Upserting vectors with metadata
4. Querying the index
5. Reading similarity scores

## Setup

### Getting a Free Pinecone API Key

1. Go to [https://www.pinecone.io/](https://www.pinecone.io/)
2. Click **Sign Up** (free tier available)
3. Create an account with email or Google/GitHub
4. Navigate to **API Keys** in the dashboard
5. Click **Create API Key**
6. Copy the API key

### Store Your API Key

Create a `.env` file in the `08-vector-databases/` directory:

```
PINECONE_API_KEY=your-api-key-here
```

Never commit `.env` files to version control.

In [ ]:
# Load environment variables
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found. Create a .env file with your key.")

print("API key loaded successfully.")

In [ ]:
# Install dependencies if needed
# !pip install pinecone python-dotenv

In [ ]:
from pinecone import Pinecone, ServerlessSpec

# Initialize Pinecone client
pc = Pinecone(api_key=PINECONE_API_KEY)

# List existing indexes
existing_indexes = pc.list_indexes().names
print(f"Existing indexes: {existing_indexes}")

## Create an Index

An index is a collection of vectors. Key parameters:
- **dimension**: Must match your embedding model output size (e.g., 1536 for OpenAI text-embedding-3-small)
- **metric**: Similarity measure (`cosine`, `dotproduct`, or `euclidean`)
- **spec**: Infrastructure configuration

In [ ]:
INDEX_NAME = "pinecone-basics-demo"
DIMENSION = 4  # Small dimension for demonstration

# Delete index if it already exists (for clean re-runs)
if INDEX_NAME in pc.list_indexes().names:
    pc.delete_index(INDEX_NAME)
    print(f"Deleted existing index: {INDEX_NAME}")

# Create new index
pc.create_index(
    name=INDEX_NAME,
    dimension=DIMENSION,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

print(f"Created index: {INDEX_NAME}")

# Connect to the index
index = pc.Index(INDEX_NAME)
print(f"Index stats: {index.describe_index_stats()}")

## Upsert Vectors with Metadata

Each vector has:
- **id**: Unique identifier
- **values**: The embedding vector (list of floats)
- **metadata**: Optional key-value pairs for filtering and context

In [ ]:
# Sample data: vectors representing short text descriptions
# In practice, you'd use an embedding model to generate these vectors

vectors = [
    {
        "id": "vec1",
        "values": [0.1, 0.2, 0.3, 0.4],
        "metadata": {
            "text": "Machine learning is a subset of artificial intelligence",
            "category": "AI",
            "difficulty": "beginner"
        }
    },
    {
        "id": "vec2",
        "values": [0.5, 0.6, 0.7, 0.8],
        "metadata": {
            "text": "Neural networks are inspired by the human brain",
            "category": "AI",
            "difficulty": "intermediate"
        }
    },
    {
        "id": "vec3",
        "values": [0.9, 0.8, 0.7, 0.6],
        "metadata": {
            "text": "Python is a popular programming language for data science",
            "category": "Programming",
            "difficulty": "beginner"
        }
    },
    {
        "id": "vec4",
        "values": [0.4, 0.3, 0.2, 0.1],
        "metadata": {
            "text": "Deep learning requires large datasets and GPU compute",
            "category": "AI",
            "difficulty": "advanced"
        }
    },
    {
        "id": "vec5",
        "values": [0.6, 0.5, 0.4, 0.3],
        "metadata": {
            "text": "SQL databases store structured data in tables",
            "category": "Databases",
            "difficulty": "beginner"
        }
    }
]

# Upsert vectors
index.upsert(vectors=vectors)
print(f"Upserted {len(vectors)} vectors.")
print(f"Index stats after upsert: {index.describe_index_stats()}")

## Query the Index

Query with a vector to find the most similar vectors in the index.

In [ ]:
# Query with a vector similar to vec1 (machine learning)
query_vector = [0.1, 0.2, 0.3, 0.5]  # Close to vec1

results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

print("Query Results:")
print("=" * 50)
for match in results["matches"]:
    print(f"ID:     {match['id']}")
    print(f"Score:  {match['score']:.4f}")
    print(f"Text:   {match['metadata']['text']}")
    print(f"Category: {match['metadata']['category']}")
    print("-" * 50)

## Reading Similarity Scores

When using cosine similarity (metric=`cosine`):
- Score ranges from 0 to 1
- **0.9 - 1.0**: Very strong match
- **0.7 - 0.9**: Strong match
- **0.5 - 0.7**: Moderate match
- **< 0.5**: Weak match

In [ ]:
# Query that should match AI-related vectors
query_vector = [0.3, 0.4, 0.5, 0.6]

results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True
)

print("Query Results with Score Interpretation:")
print("=" * 60)
for match in results["matches"]:
    score = match["score"]
    if score >= 0.9:
        strength = "Very strong match"
    elif score >= 0.7:
        strength = "Strong match"
    elif score >= 0.5:
        strength = "Moderate match"
    else:
        strength = "Weak match"
    
    print(f"ID: {match['id']} | Score: {score:.4f} ({strength})")
    print(f"   Text: {match['metadata']['text']}")
    print()

## Metadata Filtering

Use metadata filters to narrow search results before or during the similarity search.

In [ ]:
# Filter: only AI category
results = index.query(
    vector=[0.2, 0.3, 0.4, 0.5],
    top_k=3,
    include_metadata=True,
    filter={"category": {"$eq": "AI"}}
)

print("Results filtered by category = AI:")
for match in results["matches"]:
    print(f"  {match['id']}: {match['metadata']['text']} (score: {match['score']:.4f})")

print()

# Filter: beginner difficulty
results = index.query(
    vector=[0.2, 0.3, 0.4, 0.5],
    top_k=3,
    include_metadata=True,
    filter={"difficulty": {"$eq": "beginner"}}
)

print("Results filtered by difficulty = beginner:")
for match in results["matches"]:
    print(f"  {match['id']}: {match['metadata']['text']} (score: {match['score']:.4f})")

## Cleanup

In [ ]:
# Delete the demo index
pc.delete_index(INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")

## Exercises

### Exercise 1: Create an Index with OpenAI Embeddings

1. Get an OpenAI API key from [platform.openai.com](https://platform.openai.com/)
2. Add `OPENAI_API_KEY=your-key` to your `.env` file
3. Use `text-embedding-3-small` (dimension=1536) to embed real text
4. Create a Pinecone index with dimension 1536
5. Upsert at least 10 documents with metadata (title, category, tags)
6. Query the index with a natural language question

### Exercise 2: Experiment with Similarity Scores

1. Upsert vectors that are close together and far apart
2. Query with different vectors and observe score changes
3. What happens when you query with a vector that is the negative of an existing vector?

### Exercise 3: Metadata Filtering Practice

1. Create metadata with nested fields: `{"author": {"name": "Alice", "affiliation": "MIT"}}`
2. Filter by nested metadata fields
3. Combine multiple filters using `$and` and `$or`

## Key Takeaways

- Pinecone is a managed service — no infrastructure to manage
- Always use environment variables for API keys
- Metadata filtering happens server-side, reducing network transfer
- Similarity scores help you determine result quality